In [3]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import GPT4All
import os

# 1. Carregar documentos em português (TXT)
loader = TextLoader("saude_publica.txt", encoding="utf-8")
documentos = loader.load()

# 2. Dividir o texto em pedaços menores
divisor = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
fragmentos = divisor.split_documents(documentos)

# 3. Gerar embeddings com modelo pré-treinado multilíngue
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 4. Criar index FAISS
vetor_store = FAISS.from_documents(fragmentos, embedding_model)

# 5. Carregar modelo LLM local (ex: GPT4All)
modelo_llm = GPT4All()

# 6. Criar pipeline de perguntas e respostas
qa_chain = RetrievalQA.from_chain_type(
    llm=modelo_llm,
    retriever=vetor_store.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

# 7. Fazer perguntas em português
while True:
    pergunta = input("\nFaça sua pergunta (ou digite 'sair'): ")
    if pergunta.lower() == "sair":
        break

    resposta = qa_chain(pergunta)
    print(f"\n🧠 Resposta: {resposta['result']}")


KeyError: 'model'